# 14 · The best frames of many videos

Triage and preparation over many videos at once, to find the frames of one actor worth building
an avatar from: her whole body, her face alone, and close-ups of her breasts and vulva (to
refine those regions of the avatar: `preprocess.regions`). And to test both steps: a stepper at the
end goes through every image looked at, with its verdicts.

| Step | What | Per |
| --- | --- | --- |
| Triage | YOLO-pose on keyframes; the body and face scores and their gates (`preprocess.quality`); Sapiens2's contact check only where they pass; each face aligned upright and recognised (`preprocess.identity`) | sample |
| The actor | a reference photo's face, or the face seen most often among the passing samples | |
| Preparation | the detector, ViTPose and MediaPipe (`preprocess.prepare`) on the stretches of samples passing as body or face, every `FRAME_STEP`-th frame; both scores again at full resolution, the identity of each piece from its faces | frame |
| Regions | an NSFW detector on every sample and prepared frame: her breasts and vulva, scored like her face (resolution, sharpness, light, hands over them, alone), identity from the frame's face or else its shot's | sample, frame |
| The best | per kind (body, face, breasts, vulva) the passing frames by score, `MIN_GAP` seconds apart, at most `MAX_PER_VIDEO` of a video, `KEEP` in all | |
| Stepper | every image, large, filtered by stage, kind, verdict and video | |

**Body score**: a weighted geometric mean of resolution (her height in the source's pixels) and
sharpness, weighing most, then light (bright, few blown-out pixels, evenly lit), pose (distance
from an A-pose), occlusion (hands and forearms over torso, head and thighs) and being alone.
**Face score**: the same for her face (its height in the source's pixels, sharpness, light,
hands over it, alone). Both have gates; a frame below any is rejected, the factors below theirs
are the reasons. **Identity** gates both: her similarity to the actor's face, where a face is
seen (her back is not rejected for it; a prepared piece takes the median of its faces).
**Region scores** (breasts, vulva): the same as the face's, for the box the detector puts around
the region inside her box; a close-up seldom shows her face, so its identity is its shot's.

Outputs go to `outputs/harvest/<RUN>/`: per video its triage, prepared pieces and every image
looked at, and `harvest.csv`, one row per image.

In [ ]:
import csv
import dataclasses
import html
import json
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import av
import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import display
from tqdm.auto import tqdm

from avatar_splats.config import load_config
from avatar_splats.data.video import frame_size, read_frames
from avatar_splats.preprocess.face import download_face_landmarker
from avatar_splats.preprocess.hands import download_hand_landmarker
from avatar_splats.preprocess.identity import (
    FaceEmbedder,
    aligned_face,
    eye_distance,
    keypoint_face_points,
    landmark_face_points,
    main_identity,
    reference_identity,
)
from avatar_splats.preprocess.prepare import (
    BodyParts,
    PreparedShot,
    Preparer,
    PrepareSettings,
    is_prepared,
)
from avatar_splats.preprocess.quality import (
    FACE_FACTOR_NAMES,
    FACTOR_NAMES,
    FaceQualitySettings,
    QualitySettings,
    face_outline,
    measure_faces,
    measure_frames,
    score_faces,
    score_frames,
)
from avatar_splats.preprocess.regions import (
    DETECTOR,
    REGION_FACTOR_NAMES,
    REGIONS,
    RegionDetector,
    RegionQualitySettings,
    detections_of,
    detections_table,
    measure_regions,
    score_regions,
    shot_identity,
    worth_segmenting,
)
from avatar_splats.preprocess.triage import (
    Triage,
    TriageModels,
    TriageSettings,
    decode_samples,
    good_stretches,
    shot_pieces,
    triage,
)

CONFIG = "../configs/default.yaml"
config = load_config(CONFIG)
device = "cuda"

## Settings

The gates are strict on purpose: from many videos only the best frames are wanted. The
thresholds were set on `water.mp4`'s samples (sunlit, seated), identity's on `water.mp4` and
`black.mp4`: one woman's faces have a median similarity of 0.4 (0.1 with her head thrown back),
two women's at most 0.2.

In [ ]:
VIDEOS = sorted(Path("../examples/alexis").glob("*.mp4"))  # relative to notebooks/
RUN = "live-performance"
REUSE = {
    "triage",
    "prepare",
    "regions",
}  # steps that start from an earlier run's results, where stored
ACTOR_REFERENCE = []  # photos of the actor's face; empty: the face seen most often
GPU_DECODE = True  # decode on the GPU's video decoder: one CPU core instead of eight for 4K
DECODERS = 3  # videos (in triage) and pieces (in preparation) decoded ahead of the GPU
FINISHERS = 4  # images saved, prepared pieces measured and scored, at once, beside the GPU

TRIAGE = TriageSettings(
    sampling="keyframes",  # or "every_n" (decodes every frame)
    every_n=25,
    max_keyframe_gap=6.0,
    max_samples=5000,
    analysis_side=640,
    pose_model="../assets/yolo/yolo11m-pose.pt",
    parts_model="facebook/sapiens2-seg-1b",  # the contact check, on samples passing the gates
    parts_size=(512, 384),
    batch=32,  # YOLO-pose and the contact check, per GPU step
    min_person=0.5,
    min_height=0.25,  # the main person's box height, in frame heights
    good_height=0.7,
    min_joints=3,  # a face alone has its nose and eyes
    joint_seen=0.5,
    edge=0.01,
    min_sharpness=0.0,  # off: sharpness is gated by the scores, the same for every video
    context=1.6,
    max_contact=0.3,
    shot_cut=0.45,
)
QUALITY = QualitySettings(
    min_confidence=0.5,  # YOLO-pose's; preparation's ViTPose uses PREPARED_CONFIDENCE
    edge=0.01,
    good_pixels=1400.0,  # her height in the source's pixels for full resolution
    good_focus=0.08,  # triage.focus for full sharpness (water.mp4: median 0.06)
    dark=0.25,  # median luminance of her body at which brightness is 0
    bright=0.55,  # and 1 (water.mp4: median 0.63)
    max_clipped=0.1,  # share blown out at which brightness is 0
    uneven=0.25,  # 10th over 90th percentile of the blurred luminance: evenness 0
    even=0.65,  # and 1 (water.mp4: median 0.50, the best 5 % above 0.65)
    pose_tolerance=1.0,  # torso lengths from an A-pose at which pose is 0 (water.mp4: 0.6)
    max_occlusion=0.25,  # share of torso, head and thighs covered at which occlusion is 0
    max_overlap=0.3,  # share of her box someone else covers at which alone is 0
    other_face=0.15,  # similarity to the actor's face at which identity is 0
    same_face=0.35,  # and 1
    weights={
        "resolution": 2.0,
        "sharpness": 2.0,
        "brightness": 1.0,
        "evenness": 1.0,
        "pose": 1.0,
        "occlusion": 1.0,
        "alone": 1.0,
    },  # fmt: skip
    gates={
        "whole": 1.0,  # every body joint placed, the box clear of the frame's edge
        "resolution": 0.5,
        "sharpness": 0.4,
        "brightness": 0.4,
        "evenness": 0.3,
        "pose": 0.3,
        "occlusion": 0.5,
        "alone": 0.9,
        "identity": 0.3,
    },  # fmt: skip
    min_score=0.6,
)
FACE_QUALITY = FaceQualitySettings(
    min_confidence=0.5,
    edge=0.01,
    good_pixels=500.0,  # the face's height in the source's pixels for full resolution
    good_focus=0.08,
    dark=0.25,
    bright=0.55,
    max_clipped=0.1,
    uneven=0.25,
    even=0.65,
    max_occlusion=0.2,  # share of the face hands and forearms cover at which occlusion is 0
    max_overlap=0.3,
    other_face=0.15,
    same_face=0.35,
    weights={
        "resolution": 2.0,
        "sharpness": 2.0,
        "brightness": 1.0,
        "evenness": 1.0,
        "occlusion": 1.0,
        "alone": 1.0,
    },  # fmt: skip
    gates={
        "found": 1.0,
        "resolution": 0.5,
        "sharpness": 0.4,
        "brightness": 0.4,
        "evenness": 0.3,
        "occlusion": 0.7,
        "alone": 0.9,
        "identity": 0.3,
    },  # fmt: skip
    min_score=0.6,
)
PREPARED_CONFIDENCE = 0.3  # ViTPose's confidence to count a keypoint as placed
MIN_EYE_PIXELS = 12  # faces with their eyes closer than this (in the image measured): no identity

FRAME_STEP = 6  # every this many frames of a passing stretch is prepared
PIECE_FRAMES = 40  # frames per piece: about 250 MB at 1920 px, DECODERS + FINISHERS + 1 held
MIN_PIECE_FRAMES = 2
MAX_DIP = 0  # rejected samples a stretch bridges
MAX_SIDE = 1920  # pixels: prepared frames' longer side at most
PREPARE = PrepareSettings(
    min_detection=0.5,
    min_iou=0.3,
    isolate_margin=0.02,
    pose_model=config.preprocess.pose_estimator,
    keypoint_seen=PREPARED_CONFIDENCE,
    min_placed=3,
    min_face_placed=5,
    pose_batch=16,
    min_keypoint=0.3,
    landmark_crop=1.6,
)

REGION_MODELS = Path("../assets/nsfw")  # the converted detector (see preprocess.regions)
REGION_BATCH = 32
REGION_CONFIDENCE = 0.4  # the detector's confidence to count a region as found
REGION_NAMES = {  # the detector's classes, without loading it
    int(k): v for k, v in json.loads((REGION_MODELS / f"{DETECTOR}.json").read_text()).items()
}


def region_quality(good_pixels):
    return RegionQualitySettings(
        edge=0.01,
        good_pixels=good_pixels,  # the region's box's longer side in the source's pixels
        good_focus=0.08,
        dark=0.25,
        bright=0.55,
        max_clipped=0.1,
        uneven=0.25,
        even=0.65,
        max_occlusion=0.3,  # share of the region the hands' boxes cover at which occlusion is 0
        # share of it Sapiens2 labels clothing at which bare is 0: half that passes (bare's
        # gate). Clothes over a region fill it; a swimsuit pulled down under bare breasts takes
        # 10-15 % of their box, and Sapiens2 labels bare labia "lower clothing" (about 6 %)
        max_clothed=0.6,
        max_overlap=0.3,
        other_face=0.15,
        same_face=0.35,
        weights={
            "resolution": 2.0,
            "sharpness": 2.0,
            "brightness": 1.0,
            "evenness": 1.0,
            "occlusion": 1.0,
            "alone": 1.0,
        },  # fmt: skip
        gates={
            "found": 1.0,
            "resolution": 0.5,
            "sharpness": 0.4,
            "brightness": 0.4,
            "evenness": 0.3,
            "occlusion": 0.7,
            "bare": 0.5,
            "alone": 0.9,
            "identity": 0.3,
        },  # fmt: skip
        min_score=0.6,
    )


REGION_QUALITY = {"breasts": region_quality(800.0), "vulva": region_quality(300.0)}
REGION_PARTS_BATCH = 8  # Sapiens2's body parts, on prepared frames whose region may pass
REGION_PARTS_MARGIN = 0.08  # their crop around her box, in shares of its larger side
KINDS = ("body", "face", *REGIONS)

KEEP = {"body": 300, "face": 300, "breasts": 300, "vulva": 300}  # kept of each kind in all
MAX_PER_VIDEO = {"body": 60, "face": 60, "breasts": 60, "vulva": 60}
MIN_GAP = 1.0  # seconds between frames of one kind kept from one video

IMAGE_WIDTH = 960  # pixels of the images saved for the stepper
FACE_IMAGE = 224  # pixels of the upright face beside them
REGION_IMAGE = 320  # pixels: the longer side of a region's close-up beside them
REGION_COLOURS = {"breasts": (255, 140, 0), "vulva": (0, 200, 255)}
IMAGE_QUALITY = 85  # JPEG

out_dir = config.outputs / "harvest" / RUN
out_dir.mkdir(parents=True, exist_ok=True)
prepared_quality = dataclasses.replace(QUALITY, min_confidence=PREPARED_CONFIDENCE)
prepared_face_quality = dataclasses.replace(FACE_QUALITY, min_confidence=PREPARED_CONFIDENCE)
timings = {}


def video_name(video):
    # folder and file: videos of different folders may share a file name
    return f"{video.parent.name}/{video.stem}"


def timed(step, started):
    timings[step] = timings.get(step, 0.0) + time.time() - started


print(f"{len(VIDEOS)} videos: " + ", ".join(video_name(v) for v in VIDEOS))

## Images

Every image looked at is saved for the stepper: the frame with her box, keypoints and face
outline, and beside it her face aligned upright (the eyes level, as the face recognition sees
it).

In [ ]:
BONES = ((0, 1), (0, 2), (1, 3), (2, 4), (5, 6), (5, 7), (7, 9), (6, 8), (8, 10), (5, 11),
         (6, 12), (11, 12), (11, 13), (13, 15), (12, 14), (14, 16))  # fmt: skip


def save_images(path, frame, box, keypoints, landmarks, min_confidence, face_points, regions):
    # the drawn frame to path, the upright face (where its points are known) to path's "-face"
    # sibling and each region found (regions: per region its box, NaN where not found) to its
    # "-<region>" sibling; returns the paths, "" where there is none
    factor = min(1.0, IMAGE_WIDTH / frame.shape[1])
    image = cv2.resize(frame, None, fx=factor, fy=factor, interpolation=cv2.INTER_AREA)
    thickness = max(1, round(image.shape[1] / 480))
    if np.isfinite(box[:4]).all():
        x0, y0, x1, y1 = (box[:4] * factor).astype(int)
        cv2.rectangle(image, (x0, y0), (x1, y1), (255, 255, 255), thickness)
    points = keypoints[:, :2] * factor
    sure = np.nan_to_num(keypoints[:, 2]) >= min_confidence
    for a, b in BONES:
        if sure[a] and sure[b]:
            cv2.line(image, points[a].astype(int), points[b].astype(int), (0, 255, 255), thickness)
    outline = face_outline(keypoints, landmarks, min_confidence)
    if outline is not None:
        polygon = [np.round(outline * factor).astype(np.int32)]
        cv2.polylines(image, polygon, True, (255, 0, 255), thickness)
    for region, region_box in regions.items():
        if np.isfinite(region_box).all():
            x0, y0, x1, y1 = (region_box * factor).astype(int)
            cv2.rectangle(image, (x0, y0), (x1, y1), REGION_COLOURS[region], thickness)
    path.parent.mkdir(parents=True, exist_ok=True)
    quality = [cv2.IMWRITE_JPEG_QUALITY, IMAGE_QUALITY]

    def write(target, picture):
        cv2.imwrite(str(target), cv2.cvtColor(picture, cv2.COLOR_RGB2BGR), quality)
        return str(target)

    written = {"image": write(path, image), "face": ""}
    if face_points is not None:
        face = aligned_face(frame, face_points, FACE_IMAGE)
        written["face"] = write(path.with_name(path.stem + "-face.jpg"), face)
    for region, region_box in regions.items():
        written[region] = ""
        if np.isfinite(region_box).all():
            x0, y0, x1, y1 = region_box
            pad = 0.15 * max(x1 - x0, y1 - y0)
            x0, y0 = max(int(x0 - pad), 0), max(int(y0 - pad), 0)
            x1 = min(int(np.ceil(x1 + pad)), frame.shape[1])
            y1 = min(int(np.ceil(y1 + pad)), frame.shape[0])
            crop = frame[y0:y1, x0:x1]
            scale = REGION_IMAGE / max(crop.shape[:2])
            crop = cv2.resize(crop, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
            written[region] = write(path.with_name(f"{path.stem}-{region}.jpg"), crop)
    return written


def faces_of(frames, keypoints, landmarks, min_confidence, keypoint_faces=True):
    # per frame its face points (MediaPipe's where found, else the keypoints'), or None, and the
    # (n, 112, 112, 3) aligned faces big enough to recognise, with their frames' indices.
    # keypoint_faces False: only faces MediaPipe found are recognised; three keypoints of a
    # face turned away or upside down align it badly, and it then looks like no one
    points, faces, rows = [], [], []
    for i, frame in enumerate(frames):
        found = None if landmarks is None else landmark_face_points(landmarks[i])
        recognisable = found is not None or keypoint_faces
        if found is None:
            found = keypoint_face_points(keypoints[i], min_confidence)
        points.append(found)
        if recognisable and found is not None and eye_distance(found) >= MIN_EYE_PIXELS:
            faces.append(aligned_face(frame, found))
            rows.append(i)
    return points, faces, rows

## Triage

Per video: the keyframes (or every `every_n`-th frame), YOLO-pose on all of them, and the body
and face measures of each sample's main person at the sample's size (`analysis_side`).
Sapiens2's contact check, the costly part, runs only on samples that pass every gate of body or
face but `alone` and `identity` (`triage`'s `prefilter`); its contact then counts with the box
overlap for `alone`. Each face big enough is aligned and recognised; who the actor is is decided
after all videos, below. The NSFW detector runs on every sample (its results are stored beside
the triage), and her breasts and vulva are measured where it finds them inside her box.

The keyframes of the next `DECODERS` videos are decoded in background threads meanwhile:
decoding 4K is CPU work, about as long as the GPU's part, and would leave the GPU idle.

In [ ]:
embedder = FaceEmbedder()
triaged_of = {}  # video name -> its triage, measures, faces and images
triage_models = None
region_detector = None
decoders = ThreadPoolExecutor(DECODERS)
savers = ThreadPoolExecutor(FINISHERS)
pending = {}
without_identity = {
    "body": dataclasses.replace(
        QUALITY, gates={k: v for k, v in QUALITY.gates.items() if k not in ("alone", "identity")}
    ),
    "face": dataclasses.replace(
        FACE_QUALITY,
        gates={k: v for k, v in FACE_QUALITY.gates.items() if k not in ("alone", "identity")},
    ),
}
for k, video in enumerate(tqdm(VIDEOS, desc="triage", unit="video")):
    for ahead in VIDEOS[k : k + DECODERS + 1]:
        if ahead not in pending:
            pending[ahead] = decoders.submit(decode_samples, ahead, TRIAGE, True, GPU_DECODE)
    started = time.time()
    name = video_name(video)
    (out_dir / name).mkdir(parents=True, exist_ok=True)
    try:
        samples = pending.pop(video).result()
    except (OSError, ValueError, IndexError) as error:  # empty, unfinished or broken files
        print(f"{name}: skipped, it cannot be read ({error})")
        continue
    scale = samples.frames[0].shape[0] / samples.video_size[1]
    body, face = {}, {}

    def prefilter(boxes, keypoints, samples=samples, scale=scale, body=body, face=face):
        zeros = np.zeros(len(boxes))
        frames = samples.frames
        body.update(measure_frames(frames, boxes, keypoints, scale, zeros, QUALITY, True))
        face.update(measure_faces(frames, keypoints, None, None, scale, zeros, FACE_QUALITY, True))
        return (
            score_frames(body, without_identity["body"])[2]
            | score_faces(face, without_identity["face"])[2]
        )

    path = out_dir / name / "triage.npz"
    stored = Triage.load(path) if "triage" in REUSE and path.exists() else None
    if stored is not None and not (
        len(stored.times) == len(samples.times) and np.allclose(stored.times, samples.times)
    ):
        print(f"{name}: the stored triage is of other samples: triaged again")
        stored = None
    if stored is not None:
        triaged = stored
        prefilter(triaged.boxes, triaged.keypoints)
    else:
        if triage_models is None:  # loaded once, for every video triaged
            triage_models = TriageModels(TRIAGE, device)
        triaged, _ = triage(
            samples, TRIAGE, device, prefilter=prefilter, progress=True, models=triage_models
        )
        triaged.save(path)
    body["overlap"] = face["overlap"] = np.maximum(triaged.overlap, np.nan_to_num(triaged.contact))
    regions_path = out_dir / name / "regions.npz"
    if "regions" in REUSE and regions_path.exists():
        detections = detections_of(np.load(regions_path)["detections"], len(samples.frames))
    else:
        if region_detector is None:  # loaded once, for triage and preparation
            region_detector = RegionDetector(REGION_MODELS, device, REGION_BATCH, REGION_CONFIDENCE)
        detections = region_detector(samples.frames, progress=True)
        np.savez_compressed(regions_path, detections=detections_table(detections))
    regions = {
        region: measure_regions(
            samples.frames, detections, REGION_NAMES, region, triaged.boxes, scale,
            np.maximum(triaged.overlap, np.nan_to_num(triaged.contact)), REGION_QUALITY[region],
        )
        for region in REGIONS
    }  # fmt: skip
    points, faces, rows = faces_of(samples.frames, triaged.keypoints, None, QUALITY.min_confidence)
    embeddings = np.full((len(samples.frames), 512), np.nan, np.float32)
    if faces:
        embeddings[rows] = embedder(np.stack(faces))
    images = [
        savers.submit(
            save_images, out_dir / name / "images" / f"triage-{i:05d}.jpg", frame,
            triaged.boxes[i], triaged.keypoints[i], None, QUALITY.min_confidence, points[i],
            {region: regions[region]["box"][i] for region in REGIONS},
        )
        for i, frame in enumerate(samples.frames)
    ]  # fmt: skip
    triaged_of[name] = {
        "triage": triaged, "body": body, "face": face, "regions": regions,
        "embeddings": embeddings, "images": images,
    }  # fmt: skip
    del samples
    timed("triage", started)
    print(
        f"{name}: {len(triaged.times)} samples, {len(rows)} faces ({time.time() - started:.0f} s)"
    )
decoders.shutdown()
del triage_models
torch.cuda.empty_cache()

## The actor

The face in `ACTOR_REFERENCE`, or else the face seen most often among all samples' recognised
faces: the one with the most others at least `same_face` alike. In a folder of several actors
that is whoever is seen most: set `ACTOR_REFERENCE` there. Below, the faces it was made of, most
alike first (they should all be her), and per video how many of its faces are hers.

In [ ]:
started = time.time()
members = []
if ACTOR_REFERENCE:
    actor = reference_identity(
        [Path(p) for p in ACTOR_REFERENCE], embedder,
        download_face_landmarker(config.preprocess.face_landmarker),
    )  # fmt: skip
else:
    candidates, owners = [], []
    for name, entry in triaged_of.items():
        for i in np.flatnonzero(np.isfinite(entry["embeddings"]).all(axis=1)):
            candidates.append(entry["embeddings"][i])
            owners.append((name, int(i)))
    if candidates:
        actor, chosen_faces = main_identity(np.array(candidates), QUALITY.same_face)
        members = [owners[m] for m in chosen_faces]
        print(f"the actor: the face of {len(members)} of {len(candidates)} recognised faces")
    else:
        actor = None
        print("no sample shows a face big enough to recognise: identity is not gated")
for name, entry in triaged_of.items():  # NaN where no face was recognised
    if actor is None:
        entry["identity"] = entry["shot_identity"] = np.full(len(entry["embeddings"]), np.nan)
        continue
    entry["identity"] = entry["embeddings"] @ actor
    # a close-up seldom shows her face: its shot's faces say who it is
    entry["shot_identity"] = shot_identity(entry["identity"], entry["triage"].shot)
    seen = np.isfinite(entry["identity"])
    hers = (entry["identity"][seen] >= QUALITY.same_face).sum()
    print(f"  {name:30} {hers} of {seen.sum()} faces hers")
if members:
    shown = sorted(members, key=lambda m: -triaged_of[m[0]]["identity"][m[1]])[:24]
    paths = [triaged_of[n]["images"][i].result()["face"] for n, i in shown]
    faces = [cv2.imread(p)[..., ::-1] for p in paths if p]
    plt.figure(figsize=(18, 2.2))
    plt.imshow(np.hstack(faces))
    plt.axis("off")
    plt.title("the actor's faces, most alike first")
    plt.show()
timed("actor", started)

## Triage verdicts

Every sample scored as body and as face, with its likeness to the actor; what the factors were
over all videos (gates dashed), and why samples were rejected.

In [ ]:
def region_fields(verdicts, written, i, why=""):
    # a record's fields of each region: its close-up, score, verdict, reasons and factors
    fields = {}
    for region, (factors, score, ok, reasons) in verdicts.items():
        fields[f"{region}_image"] = written.get(region, "")
        fields[f"{region}_score"] = float(score[i])
        fields[f"{region}_verdict"] = "passed" if ok[i] and not why else "rejected"
        fields[f"{region}_reasons"] = ", ".join(filter(None, [why, reasons[i]]))
        fields.update({f"{region}_{k}": float(factors[k][i]) for k in REGION_FACTOR_NAMES})
    return fields


records = []  # one per image looked at
for name, entry in triaged_of.items():
    triaged, body, face = entry["triage"], entry["body"], entry["face"]
    body["identity"] = face["identity"] = entry["identity"]
    body_factors, body_score, body_ok, body_why = score_frames(body, QUALITY)
    face_factors, face_score, face_ok, face_why = score_faces(face, FACE_QUALITY)
    entry.update(body_ok=body_ok, body_score=body_score, face_ok=face_ok, face_score=face_score)
    region_verdicts = {}
    for region in REGIONS:
        entry["regions"][region]["identity"] = entry["shot_identity"]
        region_verdicts[region] = score_regions(entry["regions"][region], REGION_QUALITY[region])
        entry[f"{region}_ok"] = region_verdicts[region][2]
        entry[f"{region}_score"] = region_verdicts[region][1]
    for i in range(len(triaged.times)):
        written = entry["images"][i].result()
        nobody = triaged.people[i] == 0
        records.append({
            "video": name, "stage": "triage", "frame": int(round(triaged.times[i] * triaged.fps)),
            "time": float(triaged.times[i]), "image": written["image"],
            "face_image": written["face"], "identity": float(entry["identity"][i]),
            "shot_identity": float(entry["shot_identity"][i]),
            "body_score": float(body_score[i]),
            "body_verdict": "passed" if body_ok[i] else "rejected",
            "body_reasons": "nobody" if nobody else body_why[i],
            "face_score": float(face_score[i]),
            "face_verdict": "passed" if face_ok[i] else "rejected",
            "face_reasons": "nobody" if nobody else face_why[i],
            **{f"body_{k}": float(body_factors[k][i]) for k in FACTOR_NAMES},
            **{f"face_{k}": float(face_factors[k][i]) for k in FACE_FACTOR_NAMES},
            **region_fields(region_verdicts, written, i),
        })  # fmt: skip
someone = [r for r in records if r["body_reasons"] != "nobody"]
for kind, names, settings in (
    ("body", FACTOR_NAMES, QUALITY),
    ("face", FACE_FACTOR_NAMES, FACE_QUALITY),
    *((region, REGION_FACTOR_NAMES, REGION_QUALITY[region]) for region in REGIONS),
):
    fig, axes = plt.subplots(1, len(names), figsize=(20, 2.4))
    for ax, factor in zip(axes, names, strict=True):
        ax.hist([r[f"{kind}_{factor}"] for r in someone], bins=20, range=(0, 1))
        if factor in settings.gates:
            ax.axvline(settings.gates[factor], color="k", ls="--")
        ax.set_title(f"{kind}: {factor}", fontsize=9)
    plt.tight_layout()
    plt.show()
    counts = {}
    for r in records:
        for reason in filter(None, r[f"{kind}_reasons"].split(", ")):
            key = reason if reason == "nobody" else reason.rsplit(" ", 1)[0]
            counts[key] = counts.get(key, 0) + 1
    passed = sum(r[f"{kind}_verdict"] == "passed" for r in records)
    print(f"{kind}: {passed} of {len(records)} samples pass; rejected for:")
    for key, count in sorted(counts.items(), key=lambda kv: -kv[1]):
        print(f"  {key:30} {count:6d} ({count / len(records):.0%})")

## Preparation

Per video, its stretches of samples passing as body or face (`good_stretches`, bridging
`MAX_DIP` rejected ones, never across a cut) cut into pieces, and the cheap models on every
`FRAME_STEP`-th frame of them at `MAX_SIDE`: the detector follows the person, ViTPose on her box
(the rest greyed), MediaPipe's face and hands. Each prepared frame is scored again as body and
as face, at full size, with ViTPose's keypoints and MediaPipe's face mesh and hands, and the
NSFW detector's breasts and vulva measured again (stored beside the piece); the piece's
identity is the median of its recognised faces (the detector follows one person through it).

The GPU's part is small (the detector and ViTPose: about 0.6 s of a 40-frame piece), the rest is
CPU work: so while the GPU prepares one piece, the next `DECODERS` are decoded and the last ones
measured, scored, drawn and saved (`FINISHERS` threads).

In [ ]:
preparer = None
prepare_timings = {}


region_parts_model = None


def regions_file(path):
    # where a piece's NSFW detections and body parts are stored
    return path.with_name(path.stem + "-regions.npz")


def piece_regions(frames, result, stored, reuse):
    # a prepared piece's NSFW detections, and Sapiens2's body parts on the frames whose region
    # may pass (None on the others): from the stored ones where reused. On the main thread:
    # neither model may be called from two threads at once
    global region_detector, region_parts_model
    count = len(frames)
    if "regions" in REUSE and reuse and stored.exists():
        with np.load(stored) as data:
            labels = [None] * count
            for i, frame_labels in zip(data["labelled"], data["labels"], strict=True):
                labels[int(i)] = frame_labels
            return detections_of(data["detections"], count), labels
    if region_detector is None:
        region_detector = RegionDetector(REGION_MODELS, device, REGION_BATCH, REGION_CONFIDENCE)
    detections = region_detector(frames)
    worth = np.flatnonzero(
        worth_segmenting(detections, REGION_NAMES, result.boxes, result.scale, REGION_QUALITY)
        & np.isfinite(result.boxes[:, :4]).all(axis=1)
    )
    labels = [None] * count
    if len(worth):
        if region_parts_model is None:
            region_parts_model = BodyParts(
                TRIAGE.parts_model, REGION_PARTS_BATCH, REGION_PARTS_MARGIN, device
            )
        found, _ = region_parts_model([frames[i] for i in worth], result.boxes[worth, :4])
        for i, frame_labels in zip(worth, found, strict=True):
            labels[i] = frame_labels
    kept = [labels[i] for i in worth]
    np.savez_compressed(
        stored, detections=detections_table(detections), labelled=worth,
        labels=np.stack(kept) if kept else np.zeros((0, 1, 1), np.uint8),
    )  # fmt: skip
    return detections, labels


def decoded(video, piece):
    # (frames scaled to MAX_SIDE, the scale): in a background thread, beside the GPU. The GPU's
    # decoder fails now and then when the GPU's memory runs short: the CPU decodes those pieces

    def read(gpu):
        return list(
            read_frames(
                video, step=FRAME_STEP, max_frames=piece.count, start=piece.first,
                max_side=MAX_SIDE, gpu=gpu,
            )
        )  # fmt: skip

    try:
        frames = read(GPU_DECODE)
    except av.error.ExternalError:
        if not GPU_DECODE:
            raise
        frames = read(False)
    return frames, frames[0].shape[1] / frame_size(video)[0]


def nobody_rows(name, piece, frames, fps):
    # the records of a piece with nobody on its picked frame
    rows = []
    for i, frame in enumerate(frames):
        frame_index = piece.first + i * FRAME_STEP
        written = save_images(
            out_dir / name / "images" / f"prepared-{frame_index:07d}.jpg", frame,
            np.full(4, np.nan), np.full((17, 3), np.nan), None, 1.0, None, {},
        )  # fmt: skip
        why = "nobody on the picked frame"
        rows.append({
            "video": name, "stage": "prepared", "frame": frame_index, "time": frame_index / fps,
            "image": written["image"], "face_image": "", "identity": float("nan"),
            "body_score": 0.0, "body_verdict": "rejected", "body_reasons": why,
            "face_score": 0.0, "face_verdict": "rejected", "face_reasons": why,
            **{f"{region}_{k}": v for region in REGIONS for k, v in (
                ("score", 0.0), ("verdict", "rejected"), ("reasons", why), ("image", ""))},
        })  # fmt: skip
    return rows


def finished(name, piece, path, frames, result, save, fps, detections, labels, shot_likeness):
    # a prepared piece's records (its images saved, and the piece): in a background thread,
    # while the GPU prepares the next piece; result None: nobody on its picked frame.
    # shot_likeness: its shot's identity, for regions where the piece shows no face
    if result is None:
        return nobody_rows(name, piece, frames, fps)
    if save:
        result.save(path)
    points, faces, _ = faces_of(
        frames, result.keypoints, result.face_landmarks, PREPARED_CONFIDENCE, keypoint_faces=False
    )
    identity = np.full(len(frames), np.nan)
    if faces and actor is not None:  # the piece's identity: the median of its faces'
        identity[:] = np.median(embedder(np.stack(faces)) @ actor)
    body = measure_frames(
        frames, result.boxes, result.keypoints, result.scale, result.overlap, prepared_quality
    )
    face = measure_faces(
        frames, result.keypoints, result.face_landmarks, result.hands, result.scale,
        result.overlap, prepared_face_quality,
    )  # fmt: skip
    body["identity"] = face["identity"] = identity
    body_factors, body_score, body_ok, body_why = score_frames(body, prepared_quality)
    face_factors, face_score, face_ok, face_why = score_faces(face, prepared_face_quality)
    region_identity = np.where(np.isfinite(identity), identity, shot_likeness)
    lost = ["not found" if result.detection[i] == 0 else "" for i in range(len(frames))]
    regions, region_verdicts = {}, {}
    for region in REGIONS:
        regions[region] = measure_regions(
            frames, detections, REGION_NAMES, region, result.boxes, result.scale,
            result.overlap, REGION_QUALITY[region], labels,
        )  # fmt: skip
        regions[region]["identity"] = region_identity
        region_verdicts[region] = score_regions(regions[region], REGION_QUALITY[region])
    rows = []
    for i, frame in enumerate(frames):
        frame_index = int(result.frame_indices[i])
        why = ""
        if result.detection[i] == 0:
            why = "not found"
        elif not result.usable[i]:
            why = "unusable: too few keypoints"
        written = save_images(
            out_dir / name / "images" / f"prepared-{frame_index:07d}.jpg", frame,
            result.boxes[i], result.keypoints[i], result.face_landmarks[i], PREPARED_CONFIDENCE,
            points[i], {region: regions[region]["box"][i] for region in REGIONS},
        )  # fmt: skip
        rows.append({
            "video": name, "stage": "prepared", "frame": frame_index, "time": frame_index / fps,
            "image": written["image"], "face_image": written["face"],
            "identity": float(identity[i]), "shot_identity": shot_likeness,
            "body_score": float(body_score[i]),
            "body_verdict": "passed" if body_ok[i] and not why else "rejected",
            "body_reasons": ", ".join(filter(None, [why, body_why[i]])),
            "face_score": float(face_score[i]),
            "face_verdict": "passed" if face_ok[i] and not why else "rejected",
            "face_reasons": ", ".join(filter(None, [why, face_why[i]])),
            **{f"body_{k}": float(body_factors[k][i]) for k in FACTOR_NAMES},
            **{f"face_{k}": float(face_factors[k][i]) for k in FACE_FACTOR_NAMES},
            **region_fields(region_verdicts, written, i, lost[i]),
            "piece": str(path), "row": i,
        })  # fmt: skip
    return rows


# Three stages at once: the next pieces decoded (DECODERS threads), this one on the GPU, the
# last ones measured, scored, drawn and saved (FINISHERS threads): all but the GPU's is CPU work.
with ThreadPoolExecutor(DECODERS) as decoder, ThreadPoolExecutor(FINISHERS) as finisher:
    for video in tqdm(
        [v for v in VIDEOS if video_name(v) in triaged_of], desc="preparation", unit="video"
    ):
        started = time.time()
        name = video_name(video)
        entry = triaged_of[name]
        triaged = entry["triage"]
        best = np.max(
            [np.where(entry[f"{kind}_ok"], entry[f"{kind}_score"], 0.0) for kind in KINDS], axis=0
        )  # each sample's best passing score; 0 where it passes as none
        stretches = good_stretches(
            triaged.times, triaged.shot, best, triaged.duration, 1e-6, MAX_DIP
        )
        pieces = shot_pieces(
            stretches, triaged.times, best, triaged.fps, FRAME_STEP, PIECE_FRAMES, MIN_PIECE_FRAMES
        )
        jobs = []
        for k, piece in enumerate(pieces):
            path = out_dir / name / f"piece-{k:03d}.npz"
            reuse = "prepare" in REUSE and is_prepared(path, piece.first, FRAME_STEP, piece.count)
            jobs.append((piece, path, reuse))
        wanted = {p for job in jobs for p in (job[1], regions_file(job[1]))}
        for stale in (out_dir / name).glob("piece-*.npz"):
            if stale not in wanted:
                stale.unlink()
        upcoming = {
            n: decoder.submit(decoded, video, jobs[n][0]) for n in range(min(DECODERS, len(jobs)))
        }
        finishing = []
        for n, (piece, path, reuse) in enumerate(tqdm(jobs, desc=name, unit="piece", leave=False)):
            frames, factor = upcoming.pop(n).result()
            if n + DECODERS < len(jobs):
                upcoming[n + DECODERS] = decoder.submit(decoded, video, jobs[n + DECODERS][0])
            if reuse:
                result = PreparedShot.load(path)
            else:
                if preparer is None:
                    preparer = Preparer(
                        PREPARE, config.preprocess.detector,
                        download_hand_landmarker(config.preprocess.hand_landmarker),
                        download_face_landmarker(config.preprocess.face_landmarker), device,
                    )  # fmt: skip
                result = preparer.prepare(
                    frames, piece.first + np.arange(len(frames)) * FRAME_STEP, factor,
                    min(piece.select, len(frames) - 1), piece.shot, piece.score, prepare_timings,
                )  # fmt: skip
            detections = labels = None
            if result is not None:  # the NSFW detector and the body parts here: on the GPU
                detections, labels = piece_regions(frames, result, regions_file(path), reuse)
            nearest = int(np.argmin(np.abs(triaged.times * triaged.fps - piece.first)))
            finishing.append(
                finisher.submit(
                    finished,
                    name,
                    piece,
                    path,
                    frames,
                    result,
                    not reuse,
                    triaged.fps,
                    detections,
                    labels,
                    float(entry["shot_identity"][nearest]),
                )
            )
            if len(finishing) > FINISHERS:  # wait: every piece queued holds its frames (250 MB)
                finishing[-FINISHERS - 1].result()
            del frames, result
        rows = [row for future in finishing for row in future.result()]  # in the pieces' order
        records.extend(rows)
        timed("preparation", started)
        print(
            f"{name}: {len(pieces)} pieces, {len(rows)} frames, "
            f"{sum(r['body_verdict'] == 'passed' for r in rows)} pass as body, "
            f"{sum(r['face_verdict'] == 'passed' for r in rows)} as face "
            f"({time.time() - started:.0f} s)"
        )
savers.shutdown()
del preparer
torch.cuda.empty_cache()
for step, seconds in sorted(prepare_timings.items(), key=lambda kv: -kv[1]):
    print(f"  {step:<28}{seconds:>8.0f} s")

## The best

Per kind (body, face, breasts, vulva), the passing prepared frames, best score first, at least `MIN_GAP` seconds
from one already kept of their video, at most `MAX_PER_VIDEO` of a video, `KEEP` in all; then
every record to `harvest.csv`.

In [ ]:
for kind in KINDS:
    kept, count_of = [], {}
    passing = [
        i for i, r in enumerate(records)
        if r["stage"] == "prepared" and r[f"{kind}_verdict"] == "passed"
    ]  # fmt: skip
    for index in sorted(passing, key=lambda i: -records[i][f"{kind}_score"]):
        r = records[index]
        if len(kept) == KEEP[kind] or count_of.get(r["video"], 0) == MAX_PER_VIDEO[kind]:
            continue
        if any(
            records[j]["video"] == r["video"] and abs(records[j]["time"] - r["time"]) < MIN_GAP
            for j in kept
        ):
            continue
        kept.append(index)
        count_of[r["video"]] = count_of.get(r["video"], 0) + 1
    for index in kept:
        records[index][f"{kind}_verdict"] = "kept"
    print(f"{kind}: {len(kept)} frames kept")
    for video, count in sorted(count_of.items(), key=lambda kv: -kv[1]):
        print(f"  {video:30} {count}")
fields = [
    "video", "stage", "frame", "time", "image", "face_image", "identity", "shot_identity",
    "body_score", "body_verdict", "body_reasons", "face_score", "face_verdict", "face_reasons",
    *(f"body_{k}" for k in FACTOR_NAMES), *(f"face_{k}" for k in FACE_FACTOR_NAMES),
    *(f"{region}_{k}" for region in REGIONS for k in ("image", "score", "verdict", "reasons")),
    *(f"{region}_{k}" for region in REGIONS for k in REGION_FACTOR_NAMES), "piece", "row",
]  # fmt: skip
with open(out_dir / "harvest.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fields, restval="")
    writer.writeheader()
    writer.writerows(records)
print(f"{len(records)} images, to {out_dir / 'harvest.csv'}")

## Stepper

Every image looked at, one at a time: filter by stage, kind and verdict, and video; sort by
time or score; step with the buttons or the slider, or play. Beside the frame, her face aligned
upright (kinds body and face) or the region's close-up (breasts, vulva); below, the verdicts and
every factor.

In [ ]:
COLOURS = {"kept": "#c99700", "passed": "#2a9d2a", "rejected": "#c0392b"}
stage_choice = widgets.Dropdown(
    options=["all", "triage", "prepared"], value="prepared", description="stage"
)
kind_choice = widgets.Dropdown(options=list(KINDS), value="body", description="kind")
verdict_choice = widgets.Dropdown(
    options=["all", "kept", "passed or kept", "rejected"], value="all", description="verdict"
)
video_choice = widgets.Dropdown(
    options=["all", *sorted({r["video"] for r in records})], description="video"
)
order_choice = widgets.Dropdown(options=["time", "score"], description="order")
position = widgets.IntSlider(min=0, max=0, description="image", continuous_update=False)
play = widgets.Play(interval=700, min=0, max=0)
widgets.jslink((play, "value"), (position, "value"))
back, forward = widgets.Button(description="◀"), widgets.Button(description="▶")
picture = widgets.Image(format="jpeg", layout=widgets.Layout(width=f"{IMAGE_WIDTH}px"))
face_picture = widgets.Image(format="jpeg", layout=widgets.Layout(width=f"{REGION_IMAGE}px"))
info = widgets.HTML()
selection = []


def chosen():
    kind = kind_choice.value
    wanted = {
        "all": {"kept", "passed", "rejected"},
        "kept": {"kept"},
        "passed or kept": {"kept", "passed"},
        "rejected": {"rejected"},
    }[verdict_choice.value]
    out = [
        r for r in records
        if stage_choice.value in ("all", r["stage"])
        and video_choice.value in ("all", r["video"])
        and r[f"{kind}_verdict"] in wanted
    ]  # fmt: skip
    if order_choice.value == "score":
        out.sort(key=lambda r: -r[f"{kind}_score"])
    return out


def verdict_html(r, kind, names):
    verdict = r[f"{kind}_verdict"]
    factors = " · ".join(
        f"{name} {r[f'{kind}_{name}']:.2f}" for name in names if f"{kind}_{name}" in r
    )
    reasons = html.escape(r[f"{kind}_reasons"]) or "—"
    return (
        f"<b>{kind}</b>: <span style='color:{COLOURS[verdict]}'><b>{verdict}</b></span>, "
        f"score {r[f'{kind}_score']:.2f}; <i>{reasons}</i><br>"
        f"<span style='font-size:85%'>{factors}</span>"
    )


def show(*_):
    if not selection:
        picture.value, face_picture.value, info.value = b"", b"", "no images chosen"
        return
    r = selection[position.value]
    picture.value = Path(r["image"]).read_bytes()
    kind = kind_choice.value
    side = r["face_image"] if kind in ("body", "face") else r.get(f"{kind}_image", "")
    face_picture.value = Path(side).read_bytes() if side else b""
    identity = "no face recognised" if np.isnan(r["identity"]) else f"{r['identity']:.2f}"
    shot = r.get("shot_identity", float("nan"))
    identity += "" if np.isnan(shot) else f" (its shot: {shot:.2f})"
    verdicts = [verdict_html(r, "body", FACTOR_NAMES), verdict_html(r, "face", FACE_FACTOR_NAMES)]
    if kind in REGIONS:
        verdicts.append(verdict_html(r, kind, REGION_FACTOR_NAMES))
    info.value = (
        f"<b>{position.value + 1} of {len(selection)}</b>: {html.escape(r['video'])}, "
        f"{r['stage']}, {r['time']:.1f} s (frame {r['frame']}); likeness to the actor {identity}"
        + "".join(f"<br>{v}" for v in verdicts)
    )


def refilter(*_):
    selection[:] = chosen()
    position.max = play.max = max(len(selection) - 1, 0)
    position.value = 0
    show()


def step(change):
    def move(_):
        if selection:
            position.value = (position.value + change) % len(selection)

    return move


for control in (stage_choice, kind_choice, verdict_choice, video_choice, order_choice):
    control.observe(refilter, "value")
position.observe(show, "value")
back.on_click(step(-1))
forward.on_click(step(1))
refilter()
display(
    widgets.VBox([
        widgets.HBox([stage_choice, kind_choice, verdict_choice]),
        widgets.HBox([video_choice, order_choice]),
        widgets.HBox([back, forward, play, position]),
        widgets.HBox([picture, face_picture]),
        info,
    ])
)  # fmt: skip

In [ ]:
for step_, seconds in timings.items():
    print(f"{step_:<14}{seconds:>8.0f} s")